# MathVisionKid — chuẩn bị toàn bộ dữ liệu trên Colab

Notebook đọc toàn bộ member ứng viên được công bố theo STATE/PROGRESS và hash, gồm mọi gói bổ sung. Gói cân bằng nhỏ cũ giữ làm lịch sử/diagnostic, không dùng thay toàn kho.

Chỉ kiểm dữ liệu: không weights, trainer, GPU hoặc huấn luyện. Đăng nhập tài khoản có quyền đọc cùng ID Drive. Payload tạm nằm trong runtime Colab, tối đa một payload cache mỗi lúc, không tải kho về máy cá nhân.

Mỗi lần xử lý một đoạn tiếp nối; chạy tiếp sẽ tới các dòng/member chưa xử lý, không chọn tập nhỏ cố định. Train/validation/test giữ split gốc; sealed/pilot/challenge và history không tự vào train. Tổng candidate chưa phải tổng train-only đã kiểm. Full archive token/linkage/split audit vẫn pending.


In [ ]:
# Chỉ thư viện tokenizer đã pin từ pipeline.
%pip -q install transformers==5.16.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 88.0 MB/s eta 0:00:00


In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from pathlib import Path
import json,sys,sqlite3,hashlib
credentials, _ = google.auth.default()
drive_api = build('drive','v3',credentials=credentials,cache_discovery=False)
ROOT = Path('/content/mathvisionkid_data_only')
ROOT.mkdir(parents=True,exist_ok=True)
exec(compile('"""Account-neutral, data-only Colab bootstrap over fixed Drive IDs and hashes.\n\nNo training, weights, sharing edits or writes to Drive. Payloads are cached only\nin the caller\'s runtime directory; this module is not executed on the PC.\n"""\nfrom collections.abc import Mapping\nfrom pathlib import Path\nimport hashlib,io,json,zipfile\n\nSTATE_ID=\'1TLoQIbrRVmgsg-2uiUujCfM3HrK62Yme\'\nPROGRESS_ID=\'1D5RvQ5-V3DtYZfGxmlmvvPlVz9I-SxeX\'\nCODE_FILES=(\'exhaustive_colab_loader.py\',\'math_data_portable_resume.py\',\'math_data_archive_audit.py\',\'math_data_candidate_stream.py\')\ndef sha(data):return hashlib.sha256(data).hexdigest()\ndef verify(data,ref):\n    if len(data)!=ref[\'bytes\'] or sha(data)!=ref[\'sha256\']:raise ValueError(\'pinned bytes/SHA256 mismatch\')\n    return data\n\nclass GoogleDriveReader:\n    def __init__(self,service):self.service=service\n    def read(self,file_id,max_bytes):\n        from googleapiclient.http import MediaIoBaseDownload\n        meta=self.service.files().get(fileId=file_id,fields=\'id,size,mimeType\',supportsAllDrives=True).execute()\n        size=int(meta.get(\'size\',-1))\n        if size<0 or size>max_bytes:raise ValueError(\'Drive raw size exceeds bounded request\')\n        output=io.BytesIO();down=MediaIoBaseDownload(output,self.service.files().get_media(fileId=file_id,supportsAllDrives=True),chunksize=4*1024*1024)\n        done=False\n        while not done:\n            _,done=down.next_chunk(num_retries=3)\n            if output.tell()>max_bytes:raise ValueError(\'download exceeds bounded request\')\n        raw=output.getvalue()\n        if len(raw)!=size:raise ValueError(\'Drive raw size changed during download\')\n        return raw\n\ndef latest_controls(reader):\n    """Mutable STATE read twice; recipe itself is pinned by its exact hash."""\n    before=reader.read(STATE_ID,16*1024*1024);state=json.loads(before)\n    pointer=state[\'latest_all_data_union_recipe\']\n    if pointer[\'progress_id\']!=PROGRESS_ID:raise ValueError(\'unexpected progress control ID\')\n    member=pointer[\'member\']\n    if not isinstance(member,str) or member.startswith(\'/\') or \'..\' in Path(member).parts:raise ValueError(\'unsafe recipe member\')\n    raw=reader.read(PROGRESS_ID,128*1024*1024)\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        if archive.namelist().count(member)!=1:raise ValueError(\'missing/ambiguous latest union recipe\')\n        data=archive.read(member)\n    if sha(data)!=pointer[\'sha256\']:raise ValueError(\'STATE recipe hash differs from PROGRESS\')\n    recipe=json.loads(data)\n    if recipe.get(\'pending_discovery\'):raise ValueError(\'unresolved declared supplements; refusing partial data\')\n    if reader.read(STATE_ID,16*1024*1024)!=before:raise ValueError(\'STATE changed during read; resume from latest controls\')\n    if state.get(\'training_started\'):raise ValueError(\'unexpected training-started state for data-only notebook\')\n    return state,recipe,pointer\n\ndef bootstrap_code(reader,state,directory):\n    """Load exact code members from latest accepted bundle; no archive extractall."""\n    runs=[r for r in state.get(\'sgk_finish_runs\',[]) if r.get(\'bundle\')]\n    if not runs:raise ValueError(\'no explicit accepted code bundle\')\n    run=runs[-1];ref=run[\'bundle\'];fid=ref.get(\'id\',ref.get(\'file_id\'))\n    if not fid:raise ValueError(\'bundle file ID required\')\n    raw=verify(reader.read(fid,ref[\'bytes\']),ref)\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    receipts=[]\n    with zipfile.ZipFile(io.BytesIO(raw)) as archive:\n        if len(archive.namelist())!=len(set(archive.namelist())):raise ValueError(\'duplicate bundle member\')\n        manifest=json.loads(archive.read(\'manifest.json\'))\n        for name in CODE_FILES:\n            path=\'code/\'+name;pin=manifest[\'members\'].get(path)\n            if not pin:raise ValueError(\'latest bundle lacks candidate reader dependency: \'+name)\n            data=verify(archive.read(path),pin)\n            (directory/name).write_bytes(data);receipts.append(dict(member=path,sha256=sha(data),bytes=len(data)))\n    return dict(bundle_id=fid,bundle_sha256=ref[\'sha256\'],code_members=receipts,training_started=False)\n\nclass LazyDriveCache(Mapping):\n    """One hash-verified payload cached at a time in ephemeral Colab storage.\n\n    source/chat members remain inside their archives. Checkpoints are separate\n    and never removed by cache eviction. No files are created on Drive.\n    """\n    def __init__(self,reader,recipe,directory):\n        self.reader=reader;self.directory=Path(directory);self.directory.mkdir(parents=True,exist_ok=True)\n        self.artifacts=recipe[\'artifacts\'];self.lookup={};self.current=None\n        for digest,ref in self.artifacts.items():\n            if digest!=ref[\'sha256\'] or not ref.get(\'file_ids\'):raise ValueError(\'hash and existing Drive ID required\')\n            self.lookup[digest]=digest\n            for fid in ref[\'file_ids\']:self.lookup[fid]=digest\n    def __iter__(self):return iter(self.lookup)\n    def __len__(self):return len(self.lookup)\n    def __getitem__(self,key):\n        digest=self.lookup[key];ref=self.artifacts[digest]\n        if self.current and self.current[0]==digest:\n            path=self.current[1]\n            if path.stat().st_size!=ref[\'bytes\'] or sha(path.read_bytes())!=digest:raise ValueError(\'cached payload changed\')\n            return path\n        data=verify(self.reader.read(ref[\'file_ids\'][0],ref[\'bytes\']),ref)\n        suffix=\'.gz\' if any(n.endswith(\'.gz\') for n in ref.get(\'file_names\',[])) else \'.zip\'\n        dest=self.directory/(digest+suffix);dest.write_bytes(data)\n        if self.current:\n            old=self.current[1]\n            if old.parent.resolve()!=self.directory.resolve():raise ValueError(\'cache deletion escaped runtime directory\')\n            old.unlink()\n        self.current=(digest,dest)\n        return dest\n\ndef scope_receipt(state,recipe,pointer):\n    return dict(accepted_cases=state[\'accepted_cases\'],declared_candidate_chat_rows=state[\'training_chat_rows\'],payload_references=len(recipe[\'artifacts\']),chat_members=len(recipe[\'chat_members\']),recipe_sha256=pointer[\'sha256\'],historical_train_only_total_verified=False,full_archive_token_split_audit_complete=False,TRAINING_DATASET_READY=False,training_started=False,selection=\'ALL_DECLARED_CANDIDATE_MEMBERS_NO_BALANCED_SUBSET\',runtime_accounts=\'Authenticate any account with access to the same existing IDs; no credentials in dataset\')\n', 'math_data_drive_bootstrap.py', 'exec'), globals())
reader = GoogleDriveReader(drive_api)
state,recipe,recipe_pointer = latest_controls(reader)
code_receipt = bootstrap_code(reader,state,ROOT/'code')
sys.path.insert(0,str(ROOT/'code'))
from math_data_candidate_stream import CandidateStream,DataOnlyConsumer
from exhaustive_colab_loader import load_qwen_tokenizer,tokenize_assistant_only
print(scope_receipt(state,recipe,recipe_pointer))


{'accepted_cases': 1629781, 'declared_candidate_chat_rows': 6501780, 'payload_references': 35, 'chat_members': 29, 'recipe_sha256': '7ef836cb65a8e5ca7c6bad5955d56d4f70b2d73fedccdb4e03a0038fe7b8e461', 'historical_train_only_total_verified': False, 'full_archive_token_split_audit_complete': False, 'TRAINING_DATASET_READY': False, 'training_started': False, 'selection': 'ALL_DECLARED_CANDIDATE_MEMBERS_NO_BALANCED_SUBSET', 'runtime_accounts': 'Authenticate any account with access to the same existing IDs; no credentials in dataset'}


Checkpoint chỉ chứa tiến độ/hash/ID. Mỗi recipe/split dùng một SQLite trong thư mục Drive gốc, chạy lại cập nhật cùng ID. Chỉ một notebook ghi mỗi lúc; SQLite không là khóa phân tán Drive. Recipe thay đổi cần chuyển checkpoint append-only, không xóa bản cũ để ép chạy.


In [ ]:
FOLDER_ID='19ZxkfKVIggdcbSfsAdCtG1ibfJ0CRN3n'
SPLIT='train'  # Chỉ đổi rõ thành validation/test khi cần kiểm split đó.
ROWS_PER_RUN=20000  # Đoạn công việc tiếp nối, không chọn tập train cố định.
TOKENIZE=True  # Chỉ tokenizer pinned, không model weights.
cache=LazyDriveCache(reader,recipe,ROOT/'payload_cache')
stream=CandidateStream(recipe,cache,split=SPLIT)
checkpoint_name=f'MathVisionKid_DATA_ONLY_{SPLIT}_{stream.identity[:24]}.sqlite'
query=f"'{FOLDER_ID}' in parents and name = '{checkpoint_name}' and trashed = false"
matches=drive_api.files().list(q=query,fields='files(id,name,size,appProperties)',supportsAllDrives=True,includeItemsFromAllDrives=True).execute().get('files',[])
if len(matches)>1:
    raise RuntimeError('Checkpoint trùng tên: giữ nguyên tất cả, cần xác định ID trước khi tiếp.')
checkpoint_ref=matches[0] if matches else None
DB=ROOT/checkpoint_name
if not DB.exists() and checkpoint_ref:
    raw_db=reader.read(checkpoint_ref['id'],int(checkpoint_ref['size']))
    pin=checkpoint_ref.get('appProperties',{}).get('data_sha256')
    if not pin or hashlib.sha256(raw_db).hexdigest()!=pin:
        raise RuntimeError('Checkpoint thiếu hoặc sai hash; không tự khởi tạo lại.')
    DB.write_bytes(raw_db)
consumer=DataOnlyConsumer(DB,stream)
tokenizer=load_qwen_tokenizer() if TOKENIZE else None
span={'rows':0,'max_sequence_tokens':0,'max_completion_tokens':0,'source_links_not_reverified_here':0}
def data_only_check(row,provenance):
    if tokenizer is not None:
        item=tokenize_assistant_only(row,tokenizer)
        span['max_sequence_tokens']=max(span['max_sequence_tokens'],len(item['input_ids']))
        span['max_completion_tokens']=max(span['max_completion_tokens'],item['completion_tokens'])
    span['rows']+=1
    if provenance.get('sourcehash_origin')!='ORIGINAL_FIELD_VERIFIED_AGAINST_PINNED_SOURCE_MEMBER':
        span['source_links_not_reverified_here']+=1
try:
    progress=consumer.consume(data_only_check,max_rows=ROWS_PER_RUN)
finally:
    consumer.close()
print({'progress':progress,'this_work_span_only':span,'full_archive_linkage_split_audit_complete':False,'training_started':False})


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

{'progress': {'schema': 'MathVisionKid.candidate_consumer_progress.v1', 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train', 'counts': {'ACKNOWLEDGED_CANDIDATE': 20000}, 'acknowledged_rows': 20000, 'complete_members': 0, 'required_members': 29, 'incomplete_case_modes': 0, 'cursor': {'member_index': 0, 'next_line': 20000, 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train'}, 'stream_exhausted': False, 'all_data_training_ready': False, 'training_started': False}, 'this_work_span_only': {'rows': 20000, 'max_sequence_tokens': 1378, 'max_completion_tokens': 1213, 'source_links_not_reverified_here': 20000}, 'full_archive_linkage_split_audit_complete': False, 'training_started': False}


In [5]:
# Lưu cùng ID sau khi xử lý thành công và SQLite đã đóng.
from googleapiclient.http import MediaFileUpload
checkpoint_sha=hashlib.sha256(DB.read_bytes()).hexdigest()
properties={'data_sha256':checkpoint_sha,'recipe_identity':stream.identity,'split':SPLIT,'purpose':'DATA_ONLY_NO_TRAINING'}
if checkpoint_ref:
    fresh=drive_api.files().get(fileId=checkpoint_ref['id'],fields='id,appProperties',supportsAllDrives=True).execute()
    if fresh.get('appProperties',{}).get('data_sha256')!=checkpoint_ref.get('appProperties',{}).get('data_sha256'):
        raise RuntimeError('Checkpoint đã được cập nhật; không ghi đè tiến độ mới.')
    checkpoint_result=drive_api.files().update(fileId=checkpoint_ref['id'],body={'appProperties':properties},media_body=MediaFileUpload(str(DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,name,size,appProperties',supportsAllDrives=True).execute()
else:
    checkpoint_result=drive_api.files().create(body={'name':checkpoint_name,'parents':[FOLDER_ID],'appProperties':properties},media_body=MediaFileUpload(str(DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,name,size,appProperties',supportsAllDrives=True).execute()
verified_db=reader.read(checkpoint_result['id'],int(checkpoint_result['size']))
if hashlib.sha256(verified_db).hexdigest()!=checkpoint_sha:
    raise RuntimeError('Raw readback checkpoint không khớp; không tuyên bố lưu thành công.')
checkpoint_ref=checkpoint_result
receipt={'checkpoint_id':checkpoint_result['id'],'checkpoint_sha256':checkpoint_sha,'progress':progress,'this_work_span_only':span,'training_started':False,'TRAINING_DATASET_READY':False,'full_archive_token_linkage_split_audit_complete':False}
(ROOT/'last_data_only_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf-8')
print(receipt)


{'checkpoint_id': '1o3Q1uXA2ZTUAIHDpLCslHalg5qCbL-gr', 'checkpoint_sha256': '7b9babd8710bb598c2ec051073dc0d34de888686de97e3610cc10f1931b4a626', 'progress': {'schema': 'MathVisionKid.candidate_consumer_progress.v1', 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train', 'counts': {'ACKNOWLEDGED_CANDIDATE': 20000}, 'acknowledged_rows': 20000, 'complete_members': 0, 'required_members': 29, 'incomplete_case_modes': 0, 'cursor': {'member_index': 0, 'next_line': 20000, 'recipe_sha256': '232ac25f68dcc4efd8c94ea5da3f4db4d942e454ef98f8a7b940a13b69a683cb', 'split': 'train'}, 'stream_exhausted': False, 'all_data_training_ready': False, 'training_started': False}, 'this_work_span_only': {'rows': 20000, 'max_sequence_tokens': 1378, 'max_completion_tokens': 1213, 'source_links_not_reverified_here': 20000}, 'training_started': False, 'TRAINING_DATASET_READY': False, 'full_archive_token_linkage_split_audit_complete': False}


Runtime mất kết nối: chạy lại để tải checkpoint đúng ID/hash và tiếp nối. stream_exhausted chỉ chứng minh đọc hết member của split/recipe này; không tự chứng minh mọi nguồn liên kết đúng hoặc các split độc lập. Cần source–chat/group/question/canonical audit theo queue trước công bố sẵn sàng train. Không đổi split hoặc bỏ bài hợp lệ để qua gate. Notebook chưa được chạy trên Colab trong phiên chuẩn bị này.


In [6]:
# DATA ONLY: deepen source/hash/link/split audit over the entire published recipe.
# Does not invoke model weights, Trainer, GPU, or training.
import io,zipfile,importlib,os
from datetime import datetime,timezone
state_all,recipe_all,pointer_all=latest_controls(reader)
assert state_all['accepted_cases']==1629852 and state_all['training_chat_rows']==6502064
audit_bundle=state_all['sgk_finish_runs'][-1]['bundle']
audit_bytes=reader.read(audit_bundle['id'],audit_bundle['bytes'])
assert hashlib.sha256(audit_bytes).hexdigest()==audit_bundle['sha256']
audit_dir=ROOT/'full_audit_code'
audit_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(audit_bytes)) as za:
    ma=json.loads(za.read('manifest.json'))
    for filename in ['exhaustive_colab_loader.py','math_data_portable_resume.py','math_data_archive_audit.py','math_data_candidate_stream.py','math_data_full_audit.py']:
        member='code/'+filename
        payload=za.read(member);pin=ma['members'][member]
        assert len(payload)==pin['bytes'] and hashlib.sha256(payload).hexdigest()==pin['sha256']
        (audit_dir/filename).write_bytes(payload)
sys.path.insert(0,str(audit_dir))
import math_data_full_audit as full_audit
AUDIT_ID=full_audit.plan_identity(recipe_all)
AUDIT_NAME=f'MathVisionKid_FULL_DATA_AUDIT_{AUDIT_ID[:24]}.sqlite'
AUDIT_DB=ROOT/AUDIT_NAME
audit_query=f"'{FOLDER_ID}' in parents and name = '{AUDIT_NAME}' and trashed = false"
audit_matches=drive_api.files().list(q=audit_query,fields='files(id,name,size,appProperties)',supportsAllDrives=True,includeItemsFromAllDrives=True).execute().get('files',[])
if len(audit_matches)>1:
    raise RuntimeError('Ambiguous audit checkpoint IDs; do not reset or overwrite.')
audit_checkpoint_ref=audit_matches[0] if audit_matches else None
if not AUDIT_DB.exists() and audit_checkpoint_ref:
    prior_audit=reader.read(audit_checkpoint_ref['id'],int(audit_checkpoint_ref['size']))
    if hashlib.sha256(prior_audit).hexdigest()!=audit_checkpoint_ref.get('appProperties',{}).get('data_sha256'):
        raise RuntimeError('Audit checkpoint hash mismatch; preserve evidence.')
    AUDIT_DB.write_bytes(prior_audit)
audit_cache=LazyDriveCache(reader,recipe_all,ROOT/'full_audit_payload_cache')
auditor=full_audit.FullArchiveAudit(AUDIT_DB,recipe_all,tokenizer)
entry=auditor.next_entry()
try:
    if entry is not None:
        result=auditor.consume(entry,audit_cache[entry['sha256']],max_rows=1000)
    else:
        result={'already_complete':True}
    audit_report=auditor.report()
finally:
    auditor.close()
print({'purpose':'DATA_PREPARATION_ONLY','runtime_accelerator':os.environ.get('COLAB_ACCELERATOR','none'),'old_20k_token_checkpoint_preserved':checkpoint_result['id'],'latest_recipe':pointer_all['sha256'],'first_deep_audit_span':result,'audit':audit_report,'training_started':False})


{'purpose': 'DATA_PREPARATION_ONLY', 'runtime_accelerator': 'none', 'old_20k_token_checkpoint_preserved': '1o3Q1uXA2ZTUAIHDpLCslHalg5qCbL-gr', 'latest_recipe': '93ec4110b9de95a84bb685ae60b1e4dc4b6df8c4fe568d88c60ca9c2e301c684', 'first_deep_audit_span': {'member_key': 'c9fa305a1bf0904b794e243ad39409d5e5616e081777e3431349365955e209d5', 'rows': 1000, 'processed': 1000, 'bounded_pause': True}, 'audit': {'schema': 'MathVisionKid.full_current_archive_audit.v1', 'identity': 'e9609c3b8626b9234f884a2fdb8372166e0fd0ffbb3a1ff210d63a4bb93715b5', 'source_versions_indexed': 1000, 'unique_chat_rows_processed': 0, 'actual_rows_by_split': {}, 'source_hash_origins': {}, 'physical_chat_occurrences_processed': 0, 'exact_duplicate_references': 0, 'members_complete': 0, 'members_expected': 247, 'all_declared_members_scanned': False, 'max_sequence_tokens': None, 'max_completion_tokens': None, 'source_linkage_unverified_rows': 0, 'incomplete_four_mode_cases': 0, 'issues': {'SOURCE_SPLIT_MISSING_OR_UNKNOWN': 1

In [7]:
# Persist audit evidence even when a format issue needs adjudication; never promote it.
from googleapiclient.http import MediaFileUpload
import math_data_archive_audit as archive_audit
with archive_audit.payload(entry,audit_cache[entry['sha256']]) as audit_source_stream:
    first_source=json.loads(next(audit_source_stream))
source_format_observation={'member':entry.get('member'),'first_source_id':first_source.get('sample_id'),'explicit_split':first_source.get('split'),'keys':sorted(first_source),'source_version_count':audit_report['source_versions_indexed'],'finding':'Source records without an explicit known split remain flagged; no default train assignment or gate bypass applied.'}
audit_sha=hashlib.sha256(AUDIT_DB.read_bytes()).hexdigest()
audit_properties={'data_sha256':audit_sha,'recipe_identity':AUDIT_ID,'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING'}
if audit_checkpoint_ref:
    fresh_audit=drive_api.files().get(fileId=audit_checkpoint_ref['id'],fields='id,appProperties',supportsAllDrives=True).execute()
    if fresh_audit.get('appProperties',{}).get('data_sha256')!=audit_checkpoint_ref.get('appProperties',{}).get('data_sha256'):
        raise RuntimeError('Audit checkpoint changed on Drive; retain local evidence, do not overwrite.')
    audit_saved=drive_api.files().update(fileId=audit_checkpoint_ref['id'],body={'appProperties':audit_properties},media_body=MediaFileUpload(str(AUDIT_DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,name,size,appProperties',supportsAllDrives=True).execute()
else:
    audit_saved=drive_api.files().create(body={'name':AUDIT_NAME,'parents':[FOLDER_ID],'appProperties':audit_properties},media_body=MediaFileUpload(str(AUDIT_DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,name,size,appProperties',supportsAllDrives=True).execute()
audit_readback=reader.read(audit_saved['id'],int(audit_saved['size']))
assert hashlib.sha256(audit_readback).hexdigest()==audit_sha
audit_checkpoint_ref=audit_saved
audit_receipt={'checkpoint_id':audit_saved['id'],'checkpoint_sha256':audit_sha,'checkpoint_bytes':len(audit_readback),'all_bytes_hash_readback_verified':True,'latest_recipe_sha256':pointer_all['sha256'],'source_format_observation':source_format_observation,'report':audit_report,'old_token_checkpoint_id':checkpoint_result['id'],'training_started':False,'TRAINING_DATASET_READY':False}
(ROOT/'last_full_audit_receipt.json').write_text(json.dumps(audit_receipt,ensure_ascii=False,indent=2),encoding='utf8')
print(audit_receipt)


{'checkpoint_id': '18vXFDzqBNxLSawKWvAUhCgYZPW2fNEb3', 'checkpoint_sha256': '33d3a07ef2bc51ded1cc1958c869c64f8a88999803e57c908ea69c9b3ca47224', 'checkpoint_bytes': 1830912, 'all_bytes_hash_readback_verified': True, 'latest_recipe_sha256': '93ec4110b9de95a84bb685ae60b1e4dc4b6df8c4fe568d88c60ca9c2e301c684', 'source_format_observation': {'member': 'source.jsonl.gz', 'first_source_id': 'exhaustive_v1_div_0_100', 'explicit_split': 'train_candidate', 'keys': ['acceptance_status', 'can_solve', 'canonical_signatures', 'case_group_id', 'case_kind', 'coverage', 'curriculum_assignment', 'evaluation_eligible', 'expected_answer', 'givens', 'grade', 'hints', 'initial_guidance', 'operation_metadata', 'problem', 'provenance', 'quality_status', 'quality_validation', 'requested_quantity', 'sample_id', 'split', 'student_work', 'topic', 'training_eligible', 'training_exclusions', 'worked_steps'], 'source_version_count': 1000, 'finding': 'Source records without an explicit known split remain flagged; no de

## Mốc kiểm dữ liệu qua MCP — 21/10/2026
Đã xác thực Drive trên CPU và kiểm tokenizer đủ20.000chat, checkpointđãlưu Drive ID1o3Q1uXA2ZTUAIHDpLCslHalg5qCbL-gr. Kiểm sâu v1 index1.000source và ghi rõ nhãnnguồn train_candidate khác nhãnchat train; checkpointv1đãlưu riêng ID18vXFDzqBNxLSawKWvAUhCgYZPW2fNEb3,khôngxóa. Cần kiểm tươngthíchschema bằngauditorv2dẫnxuấttrướckhikiểm toànbộ247members. Chưa train/GPU/weights;TRAINING_DATASET_READY=false. Toànbộ71bàimới/284chatđãkiểm riêng vàcôngbốtrênDrive; không dùng20Kđểthay thếtoànkho.

In [8]:
# DATA ONLY v2: explicit source-pool label validation, no dataset rewriting.
import base64,importlib
audit_code_v2=base64.b64decode("IiIiRGVyaXZlZCB2MiBjdXJyZW50LWNhbmRpZGF0ZSBDUFUgYXVkaXQuIE5vIHRyYWluaW5nLCB3ZWlnaHRzLCBuZXR3b3JrIG9yIGRlbGV0aW9ucy4NCg0KVXNlIG9ubHkgaW4gYSBjbG91ZC9ydW50aW1lIGRpcmVjdG9yeSBmb3IgdGhlIGNvcnB1cy4gQWNjb3VudCBhbmQgbW91bnRlZCBwYXRocw0KYXJlIG91dHNpZGUgY29udGVudCBpZGVudGl0eS4gRXZlcnkgZGVjbGFyZWQgc291cmNlIG1lbWJlciBpcyBpbmRleGVkIGJlZm9yZQ0KY2hhdHM7IGFsbCBleHBsaWNpdCBzcGxpdHMgYXJlIGNoZWNrZWQgdG9nZXRoZXIuIE9yaWdpbmFsIGJ5dGVzIG5ldmVyIGNoYW5nZS4NCiIiIg0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyDQppbXBvcnQganNvbixzcWxpdGUzLGhhc2hsaWINCmltcG9ydCBtYXRoX2RhdGFfYXJjaGl2ZV9hdWRpdCBhcyBBDQppbXBvcnQgZXhoYXVzdGl2ZV9jb2xhYl9sb2FkZXIgYXMgTA0KZnJvbSBtYXRoX2RhdGFfY2FuZGlkYXRlX3N0cmVhbSBpbXBvcnQgbWVtYmVyX2tleSxjb250ZW50X2lkZW50aXR5LFNQTElUUyxfY2hlY2tfZW50cnkNCmZyb20gbWF0aF9kYXRhX3BvcnRhYmxlX3Jlc3VtZSBpbXBvcnQgY2Fub25pY2FsLGRpZ2VzdA0KDQpERVJJVkFUSU9OX1BST1ZFTkFOQ0UgPSB7DQogICAgJ3BhcmVudF9maWxlJzogJ21hdGhfZGF0YV9mdWxsX2F1ZGl0LnB5JywNCiAgICAncGFyZW50X3NoYTI1Nic6ICc2NDIzNDM5MzdjNTRjMWM3NjFlYzM4NzA4ZDc2ZmI0NmUyNjE2NTU2M2FiMjA0NWU5YmM2YzQ1YWRmMDYyNGExJywNCiAgICAnc291cmNlX2xhYmVsX2V2aWRlbmNlX2ZpbGUnOiAnZXhoYXVzdGl2ZV9wcmltYXJ5X21hdGgucHknLA0KICAgICdzb3VyY2VfbGFiZWxfZXZpZGVuY2Vfc2hhMjU2JzogJ2QwYWYxZTRhZGY3M2E1NWI4NDE2NzU3ZWU0M2RlOTAwNDc0ZjM3OGFmN2ZmODI5MGZmZmRkMGIwMTVhY2I4YjInLA0KICAgICdzb3VyY2VfbGFiZWxfZXZpZGVuY2UnOiAnbWFrZV9yb3cgcmV0dXJucyB0cmFpbl9jYW5kaWRhdGU7IGNoYXRfcmVjb3JkcyBleHBsaWNpdGx5IHJldHVybnMgdHJhaW4uJywNCiAgICAncmF3X3NvdXJjZV9yZXdyaXR0ZW4nOiBGYWxzZSwNCiAgICAnY2FuZGlkYXRlX3Bvb2xfcHJvbW90ZWRfdG9fdHJhaW4nOiBGYWxzZSwNCiAgICAndjFfY2hlY2twb2ludF9hbmRfaXNzdWVzX3ByZXNlcnZlZCc6IFRydWUsDQp9DQpTT1VSQ0VfTEFCRUxTID0gZnJvemVuc2V0KFNQTElUUykgfCB7J3RyYWluX2NhbmRpZGF0ZSd9DQoNCmRlZiBzb3VyY2VfY2hhdF9zcGxpdF9jb21wYXRpYmxlKHNvdXJjZV9sYWJlbCwgY2hhdF9zcGxpdCk6DQogICAgIyBUaGlzIGlzIGEgYnVpbGRlci1iYWNrZWQgY29tcGF0aWJpbGl0eSBjaGVjaywgbm90IGEgc291cmNlIHNwbGl0IHJld3JpdGUuDQogICAgIyBVbmtub3duL21pc3Npbmcgc291cmNlIGxhYmVscyBzdGF5IHVudmVyaWZpZWQ7IGNoYXRzIHN0aWxsIHJlcXVpcmUgU1BMSVRTLg0KICAgIHJldHVybiAoc291cmNlX2xhYmVsID09ICd0cmFpbl9jYW5kaWRhdGUnIGFuZCBjaGF0X3NwbGl0ID09ICd0cmFpbicpIG9yICgNCiAgICAgICAgc291cmNlX2xhYmVsIGluIFNQTElUUyBhbmQgc291cmNlX2xhYmVsID09IGNoYXRfc3BsaXQpDQoNCmRlZiByYXdoYXNoKGxpbmUpOnJldHVybiBoYXNobGliLnNoYTI1NihsaW5lLnJlbW92ZXN1ZmZpeCgnXG4nKS5yZW1vdmVzdWZmaXgoJ1xyJykuZW5jb2RlKCd1dGY4JykpLmhleGRpZ2VzdCgpDQpkZWYgcGxhbl9pZGVudGl0eShyZWNpcGUpOg0KICAgIHJldHVybiBkaWdlc3QoZGljdChzY2hlbWE9J2Z1bGxfY3VycmVudF9hcmNoaXZlX2F1ZGl0LnYyJyxzb3VyY2VfbWVtYmVycz1bY29udGVudF9pZGVudGl0eSh4KSBmb3IgeCBpbiByZWNpcGVbJ3NvdXJjZV9tZW1iZXJzJ11dLGNoYXRfbWVtYmVycz1bY29udGVudF9pZGVudGl0eSh4KSBmb3IgeCBpbiByZWNpcGVbJ2NoYXRfbWVtYmVycyddXSx0b2tlbml6ZXJfcmV2aXNpb249TC5SRVZJU0lPTixzb3VyY2VfbGFiZWxfcG9saWN5PSdidWlsZGVyX3Bpbm5lZF9jYW5kaWRhdGVfcG9vbF90b190cmFpbl9jaGF0X29ubHkudjEnKSkNCmNsYXNzIEZ1bGxBcmNoaXZlQXVkaXQ6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsZGF0YWJhc2UscmVjaXBlLHRva2VuaXplcixzZXF1ZW5jZV9saW1pdD00MDk2MCk6DQogICAgICAgIGlmIHJlY2lwZS5nZXQoJ3BlbmRpbmdfZGlzY292ZXJ5Jyk6cmFpc2UgVmFsdWVFcnJvcignVW5yZXNvbHZlZCBtZW1iZXIgZGlzY292ZXJ5JykNCiAgICAgICAgaWYgdG9rZW5pemVyIGlzIE5vbmU6cmFpc2UgVmFsdWVFcnJvcignQWN0dWFsIHRva2VuaXplciByZXF1aXJlZCwgbm8gZnVsbCB0b2tlbiBhdWRpdCB3aXRob3V0IGl0JykNCiAgICAgICAgc2VsZi5yZWNpcGU9cmVjaXBlO3NlbGYudG9rZW5pemVyPXRva2VuaXplcjtzZWxmLmxpbWl0PXNlcXVlbmNlX2xpbWl0DQogICAgICAgIHNlbGYuZW50cmllcz1yZWNpcGVbJ3NvdXJjZV9tZW1iZXJzJ10rcmVjaXBlWydjaGF0X21lbWJlcnMnXQ0KICAgICAgICBmb3IgZSBpbiBzZWxmLmVudHJpZXM6X2NoZWNrX2VudHJ5KGUpDQogICAgICAgIHNlbGYua2V5cz1bbWVtYmVyX2tleShlKSBmb3IgZSBpbiBzZWxmLmVudHJpZXNdDQogICAgICAgIGlmIGxlbihzZXQoc2VsZi5rZXlzKSkhPWxlbihzZWxmLmtleXMpOnJhaXNlIFZhbHVlRXJyb3IoJ0R1cGxpY2F0ZSBleGFjdCBtZW1iZXIga2V5JykNCiAgICAgICAgc2VsZi5pZGVudGl0eT1wbGFuX2lkZW50aXR5KHJlY2lwZSk7c2VsZi5kYj1zcWxpdGUzLmNvbm5lY3QoZGF0YWJhc2UpDQogICAgICAgICMgUmVmdXNlIGFuIG9sZCBpZGVudGl0eSBiZWZvcmUgYW55IHNjaGVtYSB3cml0ZSwgcHJlc2VydmluZyBpdHMgaXNzdWUgbG9nLg0KICAgICAgICBpZiBzZWxmLmRiLmV4ZWN1dGUoIlNFTEVDVCAxIEZST00gc3FsaXRlX21hc3RlciBXSEVSRSB0eXBlPSd0YWJsZScgQU5EIG5hbWU9J2NvbmZpZyciKS5mZXRjaG9uZSgpOg0KICAgICAgICAgICAgcHJpb3I9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgaWRlbnRpdHksc2VxdWVuY2VfbGltaXQgRlJPTSBjb25maWcnKS5mZXRjaG9uZSgpDQogICAgICAgICAgICBpZiBwcmlvciBhbmQgcHJpb3IhPShzZWxmLmlkZW50aXR5LHNlbGYubGltaXQpOg0KICAgICAgICAgICAgICAgIHNlbGYuZGIuY2xvc2UoKTtyYWlzZSBWYWx1ZUVycm9yKCdEaWZmZXJlbnQgaW1tdXRhYmxlIGF1ZGl0IGlkZW50aXR5OyBwcmVzZXJ2ZSBwcmlvciBjaGVja3BvaW50JykNCiAgICAgICAgc2VsZi5kYi5leGVjdXRlc2NyaXB0KCIiIg0KICAgICAgICBDUkVBVEUgVEFCTEUgSUYgTk9UIEVYSVNUUyBjb25maWcoaWRlbnRpdHkgVEVYVCBQUklNQVJZIEtFWSwgc2VxdWVuY2VfbGltaXQgSU5URUdFUik7DQogICAgICAgIENSRUFURSBUQUJMRSBJRiBOT1QgRVhJU1RTIG1lbWJlcnMoaWQgVEVYVCBQUklNQVJZIEtFWSwgcm9sZSBURVhULCBuZXh0X2xpbmUgSU5URUdFUiwgY29tcGxldGUgSU5URUdFUiwgZXhwZWN0ZWRfcm93cyBJTlRFR0VSKTsNCiAgICAgICAgQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgc291cmNlX3ZlcnNpb25zKGlkIFRFWFQscmF3X3NoYSBURVhULGNhbm9uaWNhbF9zaGEgVEVYVCxwcm9qZWN0aW9uIFRFWFQsc2lnbmF0dXJlcyBURVhULHNvdXJjZV9zcGxpdCBURVhULFBSSU1BUlkgS0VZKGlkLHJhd19zaGEpKTsNCiAgICAgICAgQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgc2FtcGxlcyhpZCBURVhUIFBSSU1BUlkgS0VZLGNhbm9uaWNhbF9zaGEgVEVYVCxjYXNlX2lkIFRFWFQsbW9kZSBURVhULHNwbGl0IFRFWFQsc291cmNlX3Jhd19zaGEgVEVYVCxsaW5rYWdlIElOVEVHRVIscmF3X2hhc2hfb3JpZ2luIFRFWFQsc2VxdWVuY2VfdG9rZW5zIElOVEVHRVIsY29tcGxldGlvbl90b2tlbnMgSU5URUdFUiwgVU5JUVVFKGNhc2VfaWQsbW9kZSkpOw0KICAgICAgICBDUkVBVEUgVEFCTEUgSUYgTk9UIEVYSVNUUyBvY2N1cnJlbmNlcyhtZW1iZXIgVEVYVCxsaW5lIElOVEVHRVIsc2FtcGxlX2lkIFRFWFQscmF3X3NoYSBURVhULGtpbmQgVEVYVCxQUklNQVJZIEtFWShtZW1iZXIsbGluZSkpOw0KICAgICAgICBDUkVBVEUgVEFCTEUgSUYgTk9UIEVYSVNUUyBzcGxpdF9rZXlzKGtpbmQgVEVYVCxrZXkgVEVYVCxzcGxpdCBURVhULFBSSU1BUlkgS0VZKGtpbmQsa2V5KSk7DQogICAgICAgIENSRUFURSBUQUJMRSBJRiBOT1QgRVhJU1RTIGlzc3VlcyhtZW1iZXIgVEVYVCxsaW5lIElOVEVHRVIsa2luZCBURVhULGRldGFpbCBURVhULFBSSU1BUlkgS0VZKG1lbWJlcixsaW5lLGtpbmQsZGV0YWlsKSk7DQogICAgICAgICIiIikNCiAgICAgICAgcHJpb3I9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgaWRlbnRpdHksc2VxdWVuY2VfbGltaXQgRlJPTSBjb25maWcnKS5mZXRjaG9uZSgpDQogICAgICAgIGlmIHByaW9yIGFuZCBwcmlvciE9KHNlbGYuaWRlbnRpdHksc2VsZi5saW1pdCk6c2VsZi5kYi5jbG9zZSgpO3JhaXNlIFZhbHVlRXJyb3IoJ0RpZmZlcmVudCBpbW11dGFibGUgYXVkaXQgaWRlbnRpdHk7IHByZXNlcnZlIHByaW9yIGNoZWNrcG9pbnQnKQ0KICAgICAgICBzZWxmLmRiLmV4ZWN1dGUoJ0lOU0VSVCBPUiBJR05PUkUgSU5UTyBjb25maWcgVkFMVUVTICg/LD8pJywoc2VsZi5pZGVudGl0eSxzZWxmLmxpbWl0KSk7c2VsZi5kYi5jb21taXQoKQ0KICAgIGRlZiBpc3N1ZShzZWxmLGtleSxsaW5lLGtpbmQsZGV0YWlsKToNCiAgICAgICAgc2VsZi5kYi5leGVjdXRlKCdJTlNFUlQgT1IgSUdOT1JFIElOVE8gaXNzdWVzIFZBTFVFUyAoPyw/LD8sPyknLChrZXksbGluZSxraW5kLHN0cihkZXRhaWwpKSkNCiAgICBkZWYgX3NvdXJjZShzZWxmLHJvdyxrZXksbGluZSx0ZXh0KToNCiAgICAgICAgaWYgcm93LmdldCgnc3BsaXQnKSBub3QgaW4gU09VUkNFX0xBQkVMUzpzZWxmLmlzc3VlKGtleSxsaW5lLCdTT1VSQ0VfU1BMSVRfTUlTU0lOR19PUl9VTktOT1dOJyxyb3cuZ2V0KCdzYW1wbGVfaWQnKSkNCiAgICAgICAgcHJvamVjdGlvbj1BLnNvdXJjZV9wcm9qZWN0aW9uKHJvdyk7aD1yYXdoYXNoKHRleHQpDQogICAgICAgIHNlbGYuZGIuZXhlY3V0ZSgnSU5TRVJUIE9SIElHTk9SRSBJTlRPIHNvdXJjZV92ZXJzaW9ucyBWQUxVRVMgKD8sPyw/LD8sPyw/KScsKHJvd1snc2FtcGxlX2lkJ10saCxkaWdlc3Qocm93KSxjYW5vbmljYWwocHJvamVjdGlvbiksY2Fub25pY2FsKEwuc291cmNlX3NpZ25hdHVyZXMocm93KSkscm93LmdldCgnc3BsaXQnKSkpDQogICAgZGVmIF9jaGF0KHNlbGYscm93LGtleSxsaW5lLHRleHQpOg0KICAgICAgICBpZiByb3cuZ2V0KCdzcGxpdCcpIG5vdCBpbiBTUExJVFM6cmFpc2UgVmFsdWVFcnJvcignRXhwbGljaXQga25vd24gY2hhdCBzcGxpdCByZXF1aXJlZDsgbm8gZGVmYXVsdCB0cmFpbicpDQogICAgICAgIEwudmFsaWRhdGVfY2hhdChyb3cpO3NpZD1MLmNhc2VfaWQocm93KTtoPWRpZ2VzdChyb3cpDQogICAgICAgIHByaW9yPXNlbGYuZGIuZXhlY3V0ZSgnU0VMRUNUIGNhbm9uaWNhbF9zaGEgRlJPTSBzYW1wbGVzIFdIRVJFIGlkPT8nLChyb3dbJ3NhbXBsZV9pZCddLCkpLmZldGNob25lKCkNCiAgICAgICAgaWYgcHJpb3I6DQogICAgICAgICAgICBraW5kPSdFWEFDVF9EVVBMSUNBVEVfUkVGRVJFTkNFJyBpZiBwcmlvclswXT09aCBlbHNlICdDT05GTElDVElOR19TQU1QTEVfSUQnDQogICAgICAgICAgICBzZWxmLmRiLmV4ZWN1dGUoJ0lOU0VSVCBJTlRPIG9jY3VycmVuY2VzIFZBTFVFUyAoPyw/LD8sPyw/KScsKGtleSxsaW5lLHJvd1snc2FtcGxlX2lkJ10scmF3aGFzaCh0ZXh0KSxraW5kKSkNCiAgICAgICAgICAgIGlmIGtpbmQhPSdFWEFDVF9EVVBMSUNBVEVfUkVGRVJFTkNFJzpzZWxmLmlzc3VlKGtleSxsaW5lLGtpbmQscm93WydzYW1wbGVfaWQnXSkNCiAgICAgICAgICAgIHJldHVybg0KICAgICAgICBjYW5kaWRhdGVzPXNlbGYuZGIuZXhlY3V0ZSgnU0VMRUNUIHJhd19zaGEscHJvamVjdGlvbixzaWduYXR1cmVzLHNvdXJjZV9zcGxpdCBGUk9NIHNvdXJjZV92ZXJzaW9ucyBXSEVSRSBpZD0/Jywoc2lkLCkpLmZldGNoYWxsKCkNCiAgICAgICAgb3JpZ2luYWw9cm93LmdldCgnc291cmNlX3Jvd19zaGEyNTYnKQ0KICAgICAgICBpZiBvcmlnaW5hbDoNCiAgICAgICAgICAgIGNhbmRpZGF0ZXM9W3MgZm9yIHMgaW4gY2FuZGlkYXRlcyBpZiBzWzBdPT1vcmlnaW5hbF0NCiAgICAgICAgICAgIGlmIG5vdCBjYW5kaWRhdGVzOnNlbGYuaXNzdWUoa2V5LGxpbmUsJ1NPVVJDRV9SQVdfSEFTSF9OT1RfRk9VTkQnLHNpZCkNCiAgICAgICAgdGFyZ2V0X21hdGNoZXM9W3MgZm9yIHMgaW4gY2FuZGlkYXRlcyBpZiBub3QgQS5saW5rYWdlX2Vycm9ycyhyb3csanNvbi5sb2FkcyhzWzFdKSldDQogICAgICAgIG1hdGNoZXM9W3MgZm9yIHMgaW4gdGFyZ2V0X21hdGNoZXMgaWYgc291cmNlX2NoYXRfc3BsaXRfY29tcGF0aWJsZShzWzNdLHJvd1snc3BsaXQnXSldDQogICAgICAgIGlmIHRhcmdldF9tYXRjaGVzIGFuZCBub3QgbWF0Y2hlczoNCiAgICAgICAgICAgIHNlbGYuaXNzdWUoa2V5LGxpbmUsJ1NPVVJDRV9DSEFUX1NQTElUX01JU01BVENIJyxjYW5vbmljYWwoZGljdChjYXNlX2lkPXNpZCxzb3VyY2VfbGFiZWxzPXNvcnRlZCh7c3RyKHNbM10pIGZvciBzIGluIHRhcmdldF9tYXRjaGVzfSksY2hhdF9zcGxpdD1yb3dbJ3NwbGl0J10pKSkNCiAgICAgICAgaWYgbGVuKG1hdGNoZXMpPT0xOg0KICAgICAgICAgICAgc291cmNlPW1hdGNoZXNbMF07b3JpZ2luPSdPUklHSU5BTF9SQVdfSEFTSF9WRVJJRklFRCcgaWYgb3JpZ2luYWwgZWxzZSAnUkVBRF9USU1FX0RFUklWRURfQllfRlVMTF9QUk9NUFRfVEFSR0VUX0xJTktBR0UnDQogICAgICAgIGVsaWYgbGVuKG1hdGNoZXMpPjE6DQogICAgICAgICAgICBzZWxmLmlzc3VlKGtleSxsaW5lLCdBTUJJR1VPVVNfU09VUkNFX1JBV19IQVNIJyxzaWQpO3NvdXJjZT1Ob25lO29yaWdpbj0nVU5WRVJJRklFRCcNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIHNlbGYuaXNzdWUoa2V5LGxpbmUsJ1NPVVJDRV9DSEFUX0xJTktBR0VfVU5WRVJJRklFRCcsc2lkKTtzb3VyY2U9Tm9uZTtvcmlnaW49J1VOVkVSSUZJRUQnDQogICAgICAgIGVuY29kZWQ9TC50b2tlbml6ZV9hc3Npc3RhbnRfb25seShyb3csc2VsZi50b2tlbml6ZXIsc2VsZi5saW1pdCkNCiAgICAgICAgcmF3X3NoYT1zb3VyY2VbMF0gaWYgc291cmNlIGVsc2UgTm9uZTtzaWduYXR1cmVzPWpzb24ubG9hZHMoc291cmNlWzJdKSBpZiBzb3VyY2UgZWxzZSByb3cuZ2V0KCdjYW5vbmljYWxfc2lnbmF0dXJlcycsW10pDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHNlbGYuZGIuZXhlY3V0ZSgnSU5TRVJUIElOVE8gc2FtcGxlcyBWQUxVRVMgKD8sPyw/LD8sPyw/LD8sPyw/LD8pJywocm93WydzYW1wbGVfaWQnXSxoLHNpZCxyb3dbJ21vZGUnXSxyb3dbJ3NwbGl0J10scmF3X3NoYSxpbnQoc291cmNlIGlzIG5vdCBOb25lKSxvcmlnaW4sbGVuKGVuY29kZWRbJ2lucHV0X2lkcyddKSxlbmNvZGVkWydjb21wbGV0aW9uX3Rva2VucyddKSkNCiAgICAgICAgZXhjZXB0IHNxbGl0ZTMuSW50ZWdyaXR5RXJyb3I6DQogICAgICAgICAgICBzZWxmLmlzc3VlKGtleSxsaW5lLCdDT05GTElDVElOR19DQVNFX01PREUnLHNpZCsnOicrcm93Wydtb2RlJ10pO3JldHVybg0KICAgICAgICBzZWxmLmRiLmV4ZWN1dGUoJ0lOU0VSVCBJTlRPIG9jY3VycmVuY2VzIFZBTFVFUyAoPyw/LD8sPyw/KScsKGtleSxsaW5lLHJvd1snc2FtcGxlX2lkJ10scmF3aGFzaCh0ZXh0KSwnQ1VSUkVOVF9DQU5ESURBVEUnKSkNCiAgICAgICAga2V5cz1bKCdncm91cCcscm93WydjYXNlX2dyb3VwX2lkJ10pLCgncXVlc3Rpb24nLEwucXVlc3Rpb25fa2V5KHJvdykpXQ0KICAgICAgICBrZXlzLmV4dGVuZCgoJ2Nhbm9uaWNhbCcscykgZm9yIHMgaW4gc2lnbmF0dXJlcykNCiAgICAgICAgZm9yIGtpbmQsdmFsdWUgaW4ga2V5czoNCiAgICAgICAgICAgIHA9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1Qgc3BsaXQgRlJPTSBzcGxpdF9rZXlzIFdIRVJFIGtpbmQ9PyBBTkQga2V5PT8nLChraW5kLHZhbHVlKSkuZmV0Y2hvbmUoKQ0KICAgICAgICAgICAgaWYgcCBhbmQgcFswXSE9cm93WydzcGxpdCddOnNlbGYuaXNzdWUoa2V5LGxpbmUsJ0NST1NTX1NQTElUXycra2luZC51cHBlcigpLHZhbHVlKQ0KICAgICAgICAgICAgc2VsZi5kYi5leGVjdXRlKCdJTlNFUlQgT1IgSUdOT1JFIElOVE8gc3BsaXRfa2V5cyBWQUxVRVMgKD8sPyw/KScsKGtpbmQsdmFsdWUscm93WydzcGxpdCddKSkNCiAgICBkZWYgY29uc3VtZShzZWxmLGVudHJ5LHBhdGgsbWF4X3Jvd3M9MjAwMDAsY29tbWl0X2V2ZXJ5PTI1Nik6DQogICAgICAgIGlmIG1heF9yb3dzIGlzIG5vdCBOb25lIGFuZCBtYXhfcm93czw9MCBvciBjb21taXRfZXZlcnk8PTA6cmFpc2UgVmFsdWVFcnJvcignUG9zaXRpdmUgd29yayBib3VuZHMgcmVxdWlyZWQnKQ0KICAgICAgICBrZXk9bWVtYmVyX2tleShlbnRyeSkNCiAgICAgICAgaWYga2V5IG5vdCBpbiBzZWxmLmtleXM6cmFpc2UgVmFsdWVFcnJvcignTWVtYmVyIG5vdCBkZWNsYXJlZCBieSBleGFjdCByZWNpcGUnKQ0KICAgICAgICBpZiBlbnRyeVsncm9sZSddPT0nY2FuZGlkYXRlX2NoYXQnOg0KICAgICAgICAgICAgbWlzc2luZz1bbWVtYmVyX2tleShlKSBmb3IgZSBpbiBzZWxmLnJlY2lwZVsnc291cmNlX21lbWJlcnMnXSBpZiBub3Qgc2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgY29tcGxldGUgRlJPTSBtZW1iZXJzIFdIRVJFIGlkPT8nLChtZW1iZXJfa2V5KGUpLCkpLmZldGNob25lKCkgb3Igbm90IHNlbGYuZGIuZXhlY3V0ZSgnU0VMRUNUIGNvbXBsZXRlIEZST00gbWVtYmVycyBXSEVSRSBpZD0/JywobWVtYmVyX2tleShlKSwpKS5mZXRjaG9uZSgpWzBdXQ0KICAgICAgICAgICAgaWYgbWlzc2luZzpyYWlzZSBWYWx1ZUVycm9yKCdJbmRleCBldmVyeSBkZWNsYXJlZCBzb3VyY2UgbWVtYmVyIGJlZm9yZSBhbnkgY2hhdHMnKQ0KICAgICAgICBjdXJzb3I9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgbmV4dF9saW5lLGNvbXBsZXRlIEZST00gbWVtYmVycyBXSEVSRSBpZD0/Jywoa2V5LCkpLmZldGNob25lKCkNCiAgICAgICAgaWYgY3Vyc29yIGFuZCBjdXJzb3JbMV06cmV0dXJuIGRpY3QoYWxyZWFkeV9jb21wbGV0ZT1UcnVlLHJvd3M9Y3Vyc29yWzBdLG1lbWJlcl9rZXk9a2V5KQ0KICAgICAgICBzdGFydD1jdXJzb3JbMF0gaWYgY3Vyc29yIGVsc2UgMDtwcm9jZXNzZWQ9MDtjb3VudD0wDQogICAgICAgIHNlbGYuZGIuZXhlY3V0ZSgnSU5TRVJUIE9SIElHTk9SRSBJTlRPIG1lbWJlcnMgVkFMVUVTICg/LD8sMCwwLD8pJywoa2V5LGVudHJ5Wydyb2xlJ10sZW50cnlbJ3Jvd3MnXSkpO3NlbGYuZGIuY29tbWl0KCkNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgd2l0aCBBLnBheWxvYWQoZW50cnkscGF0aCkgYXMgc3RyZWFtOg0KICAgICAgICAgICAgICAgIGZvciBjb3VudCx0ZXh0IGluIGVudW1lcmF0ZShzdHJlYW0sMSk6DQogICAgICAgICAgICAgICAgICAgIGlmIGNvdW50PD1zdGFydDpjb250aW51ZQ0KICAgICAgICAgICAgICAgICAgICBpZiBub3QgdGV4dC5zdHJpcCgpOnJhaXNlIFZhbHVlRXJyb3IoJ0JsYW5rIEpTT05MIHJvdycpDQogICAgICAgICAgICAgICAgICAgIHJvdz1qc29uLmxvYWRzKHRleHQpDQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJvdyxkaWN0KTpyYWlzZSBWYWx1ZUVycm9yKCdKU09OTCBvYmplY3QgcmVxdWlyZWQnKQ0KICAgICAgICAgICAgICAgICAgICBpZiBlbnRyeVsncm9sZSddPT0nY2FuZGlkYXRlX3NvdXJjZSc6c2VsZi5fc291cmNlKHJvdyxrZXksY291bnQsdGV4dCkNCiAgICAgICAgICAgICAgICAgICAgZWxzZTpzZWxmLl9jaGF0KHJvdyxrZXksY291bnQsdGV4dCkNCiAgICAgICAgICAgICAgICAgICAgc2VsZi5kYi5leGVjdXRlKCdVUERBVEUgbWVtYmVycyBTRVQgbmV4dF9saW5lPT8gV0hFUkUgaWQ9PycsKGNvdW50LGtleSkpO3Byb2Nlc3NlZCs9MQ0KICAgICAgICAgICAgICAgICAgICBpZiBwcm9jZXNzZWQlY29tbWl0X2V2ZXJ5PT0wOnNlbGYuZGIuY29tbWl0KCkNCiAgICAgICAgICAgICAgICAgICAgaWYgbWF4X3Jvd3MgaXMgbm90IE5vbmUgYW5kIHByb2Nlc3NlZD49bWF4X3Jvd3M6DQogICAgICAgICAgICAgICAgICAgICAgICBzZWxmLmRiLmNvbW1pdCgpO3JldHVybiBkaWN0KG1lbWJlcl9rZXk9a2V5LHJvd3M9Y291bnQscHJvY2Vzc2VkPXByb2Nlc3NlZCxib3VuZGVkX3BhdXNlPVRydWUpDQogICAgICAgICAgICBpZiBjb3VudCE9ZW50cnlbJ3Jvd3MnXSBvciBzdGFydD5jb3VudDpyYWlzZSBWYWx1ZUVycm9yKCdFeGFjdCBkZWNsYXJlZCBtZW1iZXIgcm93IGNvdW50IGRpZmZlcnMnKQ0KICAgICAgICAgICAgc2VsZi5kYi5leGVjdXRlKCdVUERBVEUgbWVtYmVycyBTRVQgbmV4dF9saW5lPT8sY29tcGxldGU9MSBXSEVSRSBpZD0/JywoY291bnQsa2V5KSk7c2VsZi5kYi5jb21taXQoKQ0KICAgICAgICAgICAgcmV0dXJuIGRpY3QobWVtYmVyX2tleT1rZXkscm93cz1jb3VudCxwcm9jZXNzZWQ9cHJvY2Vzc2VkLGNvbXBsZXRlPVRydWUpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICBzZWxmLmRiLnJvbGxiYWNrKCk7cmFpc2UNCiAgICBkZWYgbmV4dF9lbnRyeShzZWxmKToNCiAgICAgICAgZm9yIGUgaW4gc2VsZi5lbnRyaWVzOg0KICAgICAgICAgICAgZG9uZT1zZWxmLmRiLmV4ZWN1dGUoJ1NFTEVDVCBjb21wbGV0ZSBGUk9NIG1lbWJlcnMgV0hFUkUgaWQ9PycsKG1lbWJlcl9rZXkoZSksKSkuZmV0Y2hvbmUoKQ0KICAgICAgICAgICAgaWYgbm90IGRvbmUgb3Igbm90IGRvbmVbMF06cmV0dXJuIGUNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICBAc3RhdGljbWV0aG9kDQogICAgZGVmIGV4dGVuZF9yZWNpcGUoZGF0YWJhc2UscHJldmlvdXNfcmVjaXBlLG5leHRfcmVjaXBlKToNCiAgICAgICAgIiIiRXhwbGljaXQgYXBwZW5kLW9ubHkgbWlncmF0aW9uOyByZXRhaW4gYWxsIHByaW9yIHNvdXJjZS9jaGF0IGV2aWRlbmNlLiIiIg0KICAgICAgICBpZiBuZXh0X3JlY2lwZS5nZXQoJ3BlbmRpbmdfZGlzY292ZXJ5Jyk6cmFpc2UgVmFsdWVFcnJvcignVW5yZXNvbHZlZCBuZXcgbWVtYmVyIGRpc2NvdmVyeScpDQogICAgICAgIGZvciBmaWVsZCBpbiAoJ3NvdXJjZV9tZW1iZXJzJywnY2hhdF9tZW1iZXJzJyk6DQogICAgICAgICAgICBvbGQ9W2NvbnRlbnRfaWRlbnRpdHkoZSkgZm9yIGUgaW4gcHJldmlvdXNfcmVjaXBlW2ZpZWxkXV0NCiAgICAgICAgICAgIG5ldz1bY29udGVudF9pZGVudGl0eShlKSBmb3IgZSBpbiBuZXh0X3JlY2lwZVtmaWVsZF1dDQogICAgICAgICAgICBpZiBsZW4obmV3KTxsZW4ob2xkKSBvciBuZXdbOmxlbihvbGQpXSE9b2xkOnJhaXNlIFZhbHVlRXJyb3IoJ09sZCBzb3VyY2UvY2hhdCBwcmVmaXhlcyBtdXN0IHJlbWFpbiBpbW11dGFibGUnKQ0KICAgICAgICBiZWZvcmUsYWZ0ZXI9cGxhbl9pZGVudGl0eShwcmV2aW91c19yZWNpcGUpLHBsYW5faWRlbnRpdHkobmV4dF9yZWNpcGUpDQogICAgICAgIGRiPXNxbGl0ZTMuY29ubmVjdChkYXRhYmFzZSkNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgd2l0aCBkYjoNCiAgICAgICAgICAgICAgICBwcmlvcj1kYi5leGVjdXRlKCdTRUxFQ1QgaWRlbnRpdHkgRlJPTSBjb25maWcnKS5mZXRjaG9uZSgpDQogICAgICAgICAgICAgICAgaWYgbm90IHByaW9yIG9yIHByaW9yWzBdIT1iZWZvcmU6cmFpc2UgVmFsdWVFcnJvcignV3JvbmcgcHJpb3IgYXVkaXQgaWRlbnRpdHknKQ0KICAgICAgICAgICAgICAgIGRiLmV4ZWN1dGUoJ0NSRUFURSBUQUJMRSBJRiBOT1QgRVhJU1RTIHJlY2lwZV9oaXN0b3J5KHByZXZpb3VzX2lkZW50aXR5IFRFWFQsbmV4dF9pZGVudGl0eSBURVhULFBSSU1BUlkgS0VZKHByZXZpb3VzX2lkZW50aXR5LG5leHRfaWRlbnRpdHkpKScpDQogICAgICAgICAgICAgICAgZGIuZXhlY3V0ZSgnSU5TRVJUIE9SIElHTk9SRSBJTlRPIHJlY2lwZV9oaXN0b3J5IFZBTFVFUyAoPyw/KScsKGJlZm9yZSxhZnRlcikpDQogICAgICAgICAgICAgICAgZGIuZXhlY3V0ZSgnVVBEQVRFIGNvbmZpZyBTRVQgaWRlbnRpdHk9PycsKGFmdGVyLCkpDQogICAgICAgIGZpbmFsbHk6ZGIuY2xvc2UoKQ0KICAgICAgICByZXR1cm4gZGljdChwcmV2aW91c19pZGVudGl0eT1iZWZvcmUsbmV4dF9pZGVudGl0eT1hZnRlcixwcmlvcl9ldmlkZW5jZV9wcmVzZXJ2ZWQ9VHJ1ZSx0cmFpbmluZ19zdGFydGVkPUZhbHNlKQ0KICAgIGRlZiByZXBvcnQoc2VsZik6DQogICAgICAgIG4sbWF4c2VxLG1heGNvbXAsdW5saW5rZWQ9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgY291bnQoKiksbWF4KHNlcXVlbmNlX3Rva2VucyksbWF4KGNvbXBsZXRpb25fdG9rZW5zKSxjb2FsZXNjZShzdW0obGlua2FnZT0wKSwwKSBGUk9NIHNhbXBsZXMnKS5mZXRjaG9uZSgpDQogICAgICAgIGNvbXBsZXRlZD1zZWxmLmRiLmV4ZWN1dGUoJ1NFTEVDVCBjb3VudCgqKSBGUk9NIG1lbWJlcnMgV0hFUkUgY29tcGxldGU9MScpLmZldGNob25lKClbMF0NCiAgICAgICAgaXNzdWVzPWRpY3Qoc2VsZi5kYi5leGVjdXRlKCdTRUxFQ1Qga2luZCxjb3VudCgqKSBGUk9NIGlzc3VlcyBHUk9VUCBCWSBraW5kJykpDQogICAgICAgIG1pc3NpbmdfbW9kZXM9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgY291bnQoKikgRlJPTSAoU0VMRUNUIGNhc2VfaWQgRlJPTSBzYW1wbGVzIEdST1VQIEJZIGNhc2VfaWQgSEFWSU5HIGNvdW50KCopIT00KScpLmZldGNob25lKClbMF0NCiAgICAgICAgc291cmNlcz1zZWxmLmRiLmV4ZWN1dGUoJ1NFTEVDVCBjb3VudCgqKSBGUk9NIHNvdXJjZV92ZXJzaW9ucycpLmZldGNob25lKClbMF0NCiAgICAgICAgYWxsX2RvbmU9Y29tcGxldGVkPT1sZW4oc2VsZi5lbnRyaWVzKQ0KICAgICAgICAjIENhbm9uaWNhbCBvdmVybGFwcyBuZWVkIGFuIGV4YWN0LWhhc2ggcG9saWN5IGFkanVkaWNhdGlvbiwgbmV2ZXIgYQ0KICAgICAgICAjIGJsYW5rZXQgYnlwYXNzLiBUaGlzIHNjYW5uZXIgZG9lcyBub3QgaW52ZW50IG93bmVyIGV4Y2VwdGlvbnMuDQogICAgICAgIGNsZWFuPW5vdCBpc3N1ZXMgYW5kIG5vdCB1bmxpbmtlZCBhbmQgbm90IG1pc3NpbmdfbW9kZXMNCiAgICAgICAgcmV0dXJuIGRpY3Qoc2NoZW1hPSdNYXRoVmlzaW9uS2lkLmZ1bGxfY3VycmVudF9hcmNoaXZlX2F1ZGl0LnYyJyxpZGVudGl0eT1zZWxmLmlkZW50aXR5LGRlcml2YXRpb25fcHJvdmVuYW5jZT1ERVJJVkFUSU9OX1BST1ZFTkFOQ0Usc291cmNlX3ZlcnNpb25zX2luZGV4ZWQ9c291cmNlcyxzb3VyY2VfbGFiZWxzX2luZGV4ZWQ9ZGljdChzZWxmLmRiLmV4ZWN1dGUoJ1NFTEVDVCBjb2FsZXNjZShzb3VyY2Vfc3BsaXQsXCc8bWlzc2luZz5cJyksY291bnQoKikgRlJPTSBzb3VyY2VfdmVyc2lvbnMgR1JPVVAgQlkgc291cmNlX3NwbGl0JykpLGNhbmRpZGF0ZV9zb3VyY2VfdHJhaW5fY2hhdF9saW5rcz1zZWxmLmRiLmV4ZWN1dGUoIlNFTEVDVCBjb3VudCgqKSBGUk9NIHNhbXBsZXMgcyBKT0lOIHNvdXJjZV92ZXJzaW9ucyB2IE9OIHMuY2FzZV9pZD12LmlkIEFORCBzLnNvdXJjZV9yYXdfc2hhPXYucmF3X3NoYSBXSEVSRSBzLmxpbmthZ2U9MSBBTkQgdi5zb3VyY2Vfc3BsaXQ9J3RyYWluX2NhbmRpZGF0ZScgQU5EIHMuc3BsaXQ9J3RyYWluJyIpLmZldGNob25lKClbMF0sdW5pcXVlX2NoYXRfcm93c19wcm9jZXNzZWQ9bixhY3R1YWxfcm93c19ieV9zcGxpdD1kaWN0KHNlbGYuZGIuZXhlY3V0ZSgnU0VMRUNUIHNwbGl0LGNvdW50KCopIEZST00gc2FtcGxlcyBHUk9VUCBCWSBzcGxpdCcpKSxzb3VyY2VfaGFzaF9vcmlnaW5zPWRpY3Qoc2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgcmF3X2hhc2hfb3JpZ2luLGNvdW50KCopIEZST00gc2FtcGxlcyBHUk9VUCBCWSByYXdfaGFzaF9vcmlnaW4nKSkscGh5c2ljYWxfY2hhdF9vY2N1cnJlbmNlc19wcm9jZXNzZWQ9c2VsZi5kYi5leGVjdXRlKCdTRUxFQ1QgY291bnQoKikgRlJPTSBvY2N1cnJlbmNlcycpLmZldGNob25lKClbMF0sZXhhY3RfZHVwbGljYXRlX3JlZmVyZW5jZXM9c2VsZi5kYi5leGVjdXRlKCJTRUxFQ1QgY291bnQoKikgRlJPTSBvY2N1cnJlbmNlcyBXSEVSRSBraW5kPSdFWEFDVF9EVVBMSUNBVEVfUkVGRVJFTkNFJyIpLmZldGNob25lKClbMF0sbWVtYmVyc19jb21wbGV0ZT1jb21wbGV0ZWQsbWVtYmVyc19leHBlY3RlZD1sZW4oc2VsZi5lbnRyaWVzKSxhbGxfZGVjbGFyZWRfbWVtYmVyc19zY2FubmVkPWFsbF9kb25lLG1heF9zZXF1ZW5jZV90b2tlbnM9bWF4c2VxLG1heF9jb21wbGV0aW9uX3Rva2Vucz1tYXhjb21wLHNvdXJjZV9saW5rYWdlX3VudmVyaWZpZWRfcm93cz11bmxpbmtlZCxpbmNvbXBsZXRlX2ZvdXJfbW9kZV9jYXNlcz1taXNzaW5nX21vZGVzLGlzc3Vlcz1pc3N1ZXMsZnVsbF9hcmNoaXZlX3Rva2VuX2xpbmthZ2Vfc3BsaXRfYXVkaXRfY29tcGxldGU9Ym9vbChhbGxfZG9uZSBhbmQgY2xlYW4pLGFsbF9yb3dzX3NlbWFudGljYWxseV9odW1hbl9yZXZpZXdlZD1GYWxzZSx3aG9sZV9jdXJyaWN1bHVtX2NlcnRpZmllZD1GYWxzZSxUUkFJTklOR19EQVRBU0VUX1JFQURZPUZhbHNlLHRyYWluaW5nX3N0YXJ0ZWQ9RmFsc2UsZ3B1X3VzZWQ9RmFsc2UsbW9kZWxfd2VpZ2h0c19sb2FkZWQ9RmFsc2UpDQogICAgZGVmIGNsb3NlKHNlbGYpOnNlbGYuZGIuY2xvc2UoKQ0KDQo=").decode('utf8')
assert hashlib.sha256(audit_code_v2.encode()).hexdigest()=='81b1e3f04c5886f78282386ecb34d47ba4ba9da63edfec8856bab82034a093ee'
(audit_dir/'math_data_full_audit_v2.py').write_text(audit_code_v2,encoding='utf8')
import math_data_full_audit_v2 as audit_v2
assert audit_v2.plan_identity(recipe_all)!=AUDIT_ID
V2_ID=audit_v2.plan_identity(recipe_all)
V2_NAME=f'MathVisionKid_FULL_DATA_AUDIT_V2_{V2_ID[:24]}.sqlite'
V2_DB=ROOT/V2_NAME
v2_matches=drive_api.files().list(q=f"'{FOLDER_ID}' in parents and name = '{V2_NAME}' and trashed = false",fields='files(id,name,size,appProperties)',supportsAllDrives=True,includeItemsFromAllDrives=True).execute().get('files',[])
if len(v2_matches)>1: raise RuntimeError('Ambiguous v2 checkpoints, preserve all')
v2_ref=v2_matches[0] if v2_matches else None
if not V2_DB.exists() and v2_ref:
 raw_v2=reader.read(v2_ref['id'],int(v2_ref['size']))
 assert hashlib.sha256(raw_v2).hexdigest()==v2_ref.get('appProperties',{}).get('data_sha256')
 V2_DB.write_bytes(raw_v2)
v2auditor=audit_v2.FullArchiveAudit(V2_DB,recipe_all,tokenizer)
v2entry=v2auditor.next_entry()
try:
 v2span=v2auditor.consume(v2entry,audit_cache[v2entry['sha256']],max_rows=1000) if v2entry else {'already_complete':True}
 v2report=v2auditor.report()
finally:
 v2auditor.close()
v2sha=hashlib.sha256(V2_DB.read_bytes()).hexdigest()
v2props={'data_sha256':v2sha,'recipe_identity':V2_ID,'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING'}
if v2_ref:
 fv2=drive_api.files().get(fileId=v2_ref['id'],fields='appProperties',supportsAllDrives=True).execute()
 assert fv2.get('appProperties',{}).get('data_sha256')==v2_ref.get('appProperties',{}).get('data_sha256')
 v2_saved=drive_api.files().update(fileId=v2_ref['id'],body={'appProperties':v2props},media_body=MediaFileUpload(str(V2_DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,size,appProperties',supportsAllDrives=True).execute()
else:
 v2_saved=drive_api.files().create(body={'name':V2_NAME,'parents':[FOLDER_ID],'appProperties':v2props},media_body=MediaFileUpload(str(V2_DB),mimetype='application/vnd.sqlite3',resumable=True),fields='id,size,appProperties',supportsAllDrives=True).execute()
v2raw=reader.read(v2_saved['id'],int(v2_saved['size']))
assert hashlib.sha256(v2raw).hexdigest()==v2sha
v2_ref=v2_saved
v2receipt={'checkpoint_id':v2_saved['id'],'checkpoint_sha256':v2sha,'checkpoint_bytes':len(v2raw),'fresh_raw_readback_verified':True,'span':v2span,'report':v2report,'v1_checkpoint_preserved':audit_saved['id'],'old_token_checkpoint_preserved':checkpoint_result['id'],'runtime_accelerator':os.environ.get('COLAB_ACCELERATOR','none'),'training_started':False,'TRAINING_DATASET_READY':False}
(ROOT/'last_full_audit_v2_receipt.json').write_text(json.dumps(v2receipt,ensure_ascii=False,indent=2),encoding='utf8')
print(v2receipt)


{'checkpoint_id': '182zLHZU0XBpTfKm7ThJpfy37GgFJ7Ob2', 'checkpoint_sha256': '5ec824151d0acfec641a1ab15bc9f8ee2ea1995c1dedd95292c44741fb9f9e1b', 'checkpoint_bytes': 1540096, 'fresh_raw_readback_verified': True, 'span': {'member_key': 'c9fa305a1bf0904b794e243ad39409d5e5616e081777e3431349365955e209d5', 'rows': 1000, 'processed': 1000, 'bounded_pause': True}, 'report': {'schema': 'MathVisionKid.full_current_archive_audit.v2', 'identity': '9139670f1e76e34e8c8ee32cc541d33399e86445cbc4da6b668b664e915655b0', 'derivation_provenance': {'parent_file': 'math_data_full_audit.py', 'parent_sha256': '642343937c54c1c761ec38708d76fb46e26165563ab2045e9bc6c45adf0624a1', 'source_label_evidence_file': 'exhaustive_primary_math.py', 'source_label_evidence_sha256': 'd0af1e4adf73a55b8416757ee43de900474f378af7ff8290fffdd0b015acb8b2', 'source_label_evidence': 'make_row returns train_candidate; chat_records explicitly returns train.', 'raw_source_rewritten': False, 'candidate_pool_promoted_to_train': False, 'v1_ch

In [9]:
# Finite CPU data audit over all247 declared source/chat members. No training.
import threading,time,traceback
from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
if globals().get('full_audit_worker') is not None and full_audit_worker.is_alive():
    raise RuntimeError('An audit worker is already running; do not launch a duplicate.')
worker_status_path=ROOT/'full_audit_v2_worker_status.json'
full_audit_worker_state={'state':'STARTING','started_utc':datetime.now(timezone.utc).isoformat(),'notebook_id':'1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l','audit_identity':V2_ID,'recipe_sha256':pointer_all['sha256'],'checkpoint_id':v2_ref['id'],'training_started':False,'gpu_used':False,'model_weights_loaded':False}
def write_worker_status():
    full_audit_worker_state['updated_utc']=datetime.now(timezone.utc).isoformat()
    temp_status=worker_status_path.with_suffix('.writing')
    temp_status.write_text(json.dumps(full_audit_worker_state,ensure_ascii=False,indent=2),encoding='utf8')
    temp_status.replace(worker_status_path)
def finite_full_data_audit_worker():
    work_api=build('drive','v3',credentials=credentials,cache_discovery=False)
    work_reader=GoogleDriveReader(work_api)
    work_cache=LazyDriveCache(work_reader,recipe_all,ROOT/'full_audit_worker_payload_cache')
    work_auditor=None
    checkpoint=dict(v2_ref)
    last_save=0.0
    def save_work_checkpoint():
        nonlocal checkpoint,last_save
        work_auditor.db.commit()
        snapshot=ROOT/'full_audit_v2_snapshot.sqlite'
        with sqlite3.connect(snapshot) as target_db:
            work_auditor.db.backup(target_db)
        hasher=hashlib.sha256()
        with snapshot.open('rb') as sf:
            for block in iter(lambda:sf.read(4*1024*1024),b''):hasher.update(block)
        snapshot_sha=hasher.hexdigest()
        remote=work_api.files().get(fileId=checkpoint['id'],fields='id,appProperties',supportsAllDrives=True).execute()
        if remote.get('appProperties',{}).get('data_sha256')!=checkpoint.get('appProperties',{}).get('data_sha256'):
            raise RuntimeError('Concurrent checkpoint update; stop without overwriting.')
        props={'data_sha256':snapshot_sha,'recipe_identity':V2_ID,'purpose':'FULL_DATA_AUDIT_ONLY_NO_TRAINING'}
        saved=work_api.files().update(fileId=checkpoint['id'],body={'appProperties':props},media_body=MediaFileUpload(str(snapshot),mimetype='application/vnd.sqlite3',resumable=True),fields='id,size,appProperties',supportsAllDrives=True).execute()
        class HashSink:
            def __init__(self):self.hasher=hashlib.sha256();self.size=0
            def write(self,data):self.hasher.update(data);self.size+=len(data);return len(data)
        sink=HashSink()
        download=MediaIoBaseDownload(sink,work_api.files().get_media(fileId=saved['id'],supportsAllDrives=True),chunksize=4*1024*1024)
        done=False
        while not done:_,done=download.next_chunk(num_retries=3)
        if sink.hasher.hexdigest()!=snapshot_sha or sink.size!=snapshot.stat().st_size:
            raise RuntimeError('Checkpoint raw streaming readback mismatch.')
        checkpoint=saved;last_save=time.monotonic()
        full_audit_worker_state['checkpoint']={'id':saved['id'],'bytes':sink.size,'sha256':snapshot_sha,'fresh_raw_readback_verified':True}
    try:
        work_auditor=audit_v2.FullArchiveAudit(V2_DB,recipe_all,tokenizer)
        full_audit_worker_state['state']='RUNNING_DATA_ONLY'
        write_worker_status()
        while True:
            next_member=work_auditor.next_entry()
            if next_member is None:break
            before_report=work_auditor.report()
            span_work=work_auditor.consume(next_member,work_cache[next_member['sha256']],max_rows=50000,commit_every=256)
            full_audit_worker_state['last_span']=span_work
            full_audit_worker_state['report']=work_auditor.report()
            full_audit_worker_state['current_member']={'role':next_member['role'],'member':next_member.get('member'),'payload_sha256':next_member['sha256']}
            if not span_work.get('processed',0) and not span_work.get('complete') and not span_work.get('already_complete'):
                raise RuntimeError('Audit made no cursor progress; refuse an endless loop.')
            if last_save==0 or time.monotonic()-last_save>=300:
                save_work_checkpoint()
            write_worker_status()
        save_work_checkpoint()
        final=work_auditor.report()
        full_audit_worker_state['report']=final
        full_audit_worker_state['state']='AUDIT_PASS_PENDING_CURRICULUM_ACCEPTANCE' if final['full_archive_token_linkage_split_audit_complete'] else 'AUDIT_FINISHED_WITH_FINDINGS'
    except Exception as exc:
        full_audit_worker_state['state']='STOPPED_REQUIRES_AUDIT_REPAIR'
        full_audit_worker_state['error_type']=type(exc).__name__
        full_audit_worker_state['error']=str(exc)
        if work_auditor is not None:
            full_audit_worker_state['report']=work_auditor.report()
            try:save_work_checkpoint()
            except Exception as checkpoint_error:
                full_audit_worker_state['checkpoint_save_error']=str(checkpoint_error)
    finally:
        if work_auditor is not None:work_auditor.close()
        write_worker_status()
write_worker_status()
full_audit_worker=threading.Thread(target=finite_full_data_audit_worker,name='MathVisionKidFiniteCPUDataAudit',daemon=True)
full_audit_worker.start()
print(full_audit_worker_state)



{'state': 'STARTING', 'started_utc': '2026-10-21T11:12:42.153326+00:00', 'notebook_id': '1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l', 'audit_identity': '9139670f1e76e34e8c8ee32cc541d33399e86445cbc4da6b668b664e915655b0', 'recipe_sha256': '93ec4110b9de95a84bb685ae60b1e4dc4b6df8c4fe568d88c60ca9c2e301c684', 'checkpoint_id': '182zLHZU0XBpTfKm7ThJpfy37GgFJ7Ob2', 'training_started': False, 'gpu_used': False, 'model_weights_loaded': False, 'updated_utc': '2026-10-21T11:12:42.154959+00:00'}


In [11]:
# Read-only progress; does not start another worker.
print(json.loads(worker_status_path.read_text(encoding='utf8')))


{'state': 'RUNNING_DATA_ONLY', 'started_utc': '2026-10-21T11:12:42.153326+00:00', 'notebook_id': '1TeCoP57miKmhESZVKWFqD-XqZmQd5u4l', 'audit_identity': '9139670f1e76e34e8c8ee32cc541d33399e86445cbc4da6b668b664e915655b0', 'recipe_sha256': '93ec4110b9de95a84bb685ae60b1e4dc4b6df8c4fe568d88c60ca9c2e301c684', 'checkpoint_id': '182zLHZU0XBpTfKm7ThJpfy37GgFJ7Ob2', 'training_started': False, 'gpu_used': False, 'model_weights_loaded': False, 'updated_utc': '2026-10-21T11:13:49.348443+00:00', 'last_span': {'member_key': '5083270150e8ada4650b74d2bc3abd2b664392d7e76f3c10be6950a79947fe09', 'rows': 10000, 'processed': 10000, 'complete': True}, 'report': {'schema': 'MathVisionKid.full_current_archive_audit.v2', 'identity': '9139670f1e76e34e8c8ee32cc541d33399e86445cbc4da6b668b664e915655b0', 'derivation_provenance': {'parent_file': 'math_data_full_audit.py', 'parent_sha256': '642343937c54c1c761ec38708d76fb46e26165563ab2045e9bc6c45adf0624a1', 'source_label_evidence_file': 'exhaustive_primary_math.py', 's